### Part 4. Langchain - Agent
- Objectives: Langchain 라이브러리를 활용하여 Agent기능 알아보기

In [ ]:
!pip install langchain pinecone-client datasets openai tiktoken

In [ ]:
# import time
# os.environ['PINECONE_API_KEY']='<YOUR_API_KEY>'
# os.environ['OPENAI_API_KEY']='<YOUR_API_KEY>'

In [25]:
import os

from dotenv import load_dotenv
load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

In [7]:
#기반 데이터셋 로드
from datasets import load_dataset
dataset = load_dataset("lcw99/wikipedia-korean-20221001", split='train[:100]')

In [4]:
!pip install langchain_text_splitters

In [8]:
# 청킹 메서드 정의
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)

In [9]:
# 임베딩 모델 정의
from langchain_openai import OpenAIEmbeddings

model_name = 'text-embedding-ada-002'

embed = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=openai_api_key
)

In [26]:
# 파인콘 인덱스 연결
from pinecone import Pinecone

# api_key = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=pinecone_api_key)

In [11]:
from time import sleep
from pinecone import ServerlessSpec

def _get_value(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def _wait_until(predicate, interval=1):
    while not predicate():
        sleep(interval)

def ensure_pinecone_index(
    pc,
    index_name: str,
    dimension: int,
    metric: str = "cosine",
    cloud: str = "aws",
    region: str = "us-east-1",
):
    spec = ServerlessSpec(cloud=cloud, region=region)
    existing = pc.list_indexes().names()

    if index_name in existing:
        desc = pc.describe_index(index_name)
        current_dimension = _get_value(desc, "dimension")

        if current_dimension != dimension:
            print(f"Recreating '{index_name}': {current_dimension} -> {dimension}")

            pc.configure_index(name=index_name, deletion_protection="disabled")
            _wait_until(
                lambda: _get_value(pc.describe_index(index_name), "deletion_protection") == "disabled"
            )

            pc.delete_index(index_name)
            _wait_until(lambda: index_name not in pc.list_indexes().names())

            pc.create_index(
                name=index_name,
                dimension=dimension,
                metric=metric,
                spec=spec,
                deletion_protection="disabled",
            )
    else:
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric=metric,
            spec=spec,
            deletion_protection="disabled",
        )

    _wait_until(lambda: _get_value(pc.describe_index(index_name), "status")["ready"])
    return pc.Index(index_name)


In [12]:
index = ensure_pinecone_index(pc, index_name="vector", dimension=1536, metric="cosine")
print(index.describe_index_stats())

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1521}},
 'total_vector_count': 1521,
 'vector_type': 'dense'}


In [13]:
#업서트 하기
from tqdm.auto import tqdm
from uuid import uuid4

def process_and_upload_records(dataset, batch_limit=100):
    texts = []
    metadatas = []

    for i, record in enumerate(tqdm(dataset)):
        metadata = {
            'id': str(record['id']),
            'source': record['url'],
            'title': record['title']
        }

        record_texts = text_splitter.split_text(record['text'])
        record_metadatas = [
            {"chunk": j, "text": text, **metadata}
            for j, text in enumerate(record_texts)
        ]

        texts.extend(record_texts)
        metadatas.extend(record_metadatas)

        if len(texts) >= batch_limit:
            upload_data(texts, metadatas)
            texts, metadatas = [], []

    if texts:
        upload_data(texts, metadatas)

def upload_data(texts, metadatas):
    ids = [str(uuid4()) for _ in range(len(texts))]
    embeds = embed.embed_documents(texts)
    index.upsert(vectors=zip(ids, embeds, metadatas))

process_and_upload_records(dataset)


100%|██████████| 100/100 [00:43<00:00,  2.32it/s]


In [15]:

index_name = 'vector'
index = pc.Index(index_name)

In [16]:
#Langchain에서 벡터스토어 연결성 맺기
from langchain_pinecone import PineconeVectorStore as Pinecone

text_field = "text"

vectorstore = Pinecone(
    index=index, embedding=embed, text_key=text_field
)

In [17]:
query = "지미 카터가 누구야?"

vectorstore.similarity_search(
    query,
    k=3
)

[Document(id='84424b7f-8456-4e2e-b5c1-78dc9002ebe2', metadata={'chunk': 0.0, 'id': '5', 'source': 'https://ko.wikipedia.org/wiki/%EC%A7%80%EB%AF%B8%20%EC%B9%B4%ED%84%B0', 'title': '지미 카터'}, page_content='제임스 얼 카터 주니어(, 1924년 10월 1일 ~ )는 민주당 출신 미국 39대 대통령 (1977년 ~ 1981년)이다.\n\n생애\n\n어린 시절 \n지미 카터는 조지아주 섬터 카운티 플레인스 마을에서 태어났다.\n\n조지아 공과대학교를 졸업하였다. 그 후 해군에 들어가 전함·원자력·잠수함의 승무원으로 일하였다. 1953년 미국 해군 대위로 예편하였고 이후 땅콩·면화 등을 가꿔 많은 돈을 벌었다. 그의 별명이 "땅콩 농부" (Peanut Farmer)로 알려졌다.'),
 Document(id='654b2170-0b1c-4b8b-b7b1-0bc5179ebb3d', metadata={'chunk': 6.0, 'id': '5', 'source': 'https://ko.wikipedia.org/wiki/%EC%A7%80%EB%AF%B8%20%EC%B9%B4%ED%84%B0', 'title': '지미 카터'}, page_content='퇴임 이후 \n\n퇴임 이후 민간 자원을 적극 활용한 비영리 기구인 카터 재단을 설립한 뒤 민주주의 실현을 위해 제 3세계의 선거 감시 활동 및 기니 벌레에 의한 드라쿤쿠르스 질병 방재를 위해 힘썼다. 미국의 빈곤층 지원 활동, 사랑의 집짓기 운동, 국제 분쟁 중재 등의 활동도 했다.\n\n카터는 카터 행정부 이후 미국이 북핵 위기, 코소보 전쟁, 이라크 전쟁과 같이 미국이 군사적 행동을 최후로 선택하는 전통적 사고를 버리고 군사적 행동을 선행하는 행위에 대해 깊은 유감을 표시 하며 미국의 군사적 활동에 강한 반대 입장을 보이고 있다.'),
 Document(id='f10b

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains.conversation.memory import ConversationBufferWindowMemory
from langchain_classic.chains import RetrievalQA

llm = ChatOpenAI(
    openai_api_key=openai_api_key,
    model_name='gpt-4.1-mini',
    temperature=0.0
)

conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=5,
    return_messages=True
)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    memory=conversational_memory,
    chain_type="stuff",
    retriever=vectorstore.as_retriever()
)

In [35]:
qa.run(query)

'지미 카터(제임스 얼 카터 주니어)는 미국의 제39대 대통령으로, 1977년부터 1981년까지 재임했습니다. 그는 조지아주 플레인스 마을에서 태어나 조지아 공과대학교를 졸업한 후 해군에서 복무했으며, 이후 땅콩 농사를 지어 큰 성공을 거두었습니다. 대통령 퇴임 후에는 카터 재단을 설립해 민주주의 실현, 국제 분쟁 중재, 빈곤층 지원 등 다양한 인도주의 활동에 힘썼습니다. 또한, 그는 미국의 군사적 행동에 대해 비판적인 입장을 보였고, 대한민국과의 관계에서도 중요한 영향을 미친 인물입니다.'

In [36]:
# 에이전트 없이 질문해보기
qa.run('오늘 야식 메뉴좀 추천해줘')

'오늘 야식으로는 간단하면서도 맛있는 메뉴를 추천해드릴게요!\n\n1. 떡볶이와 튀김 세트 – 매콤달콤한 떡볶이와 바삭한 튀김 조합은 언제나 인기죠.\n2. 치킨과 맥주 – 치킨은 야식의 정석! 맥주와 함께라면 더 좋고요.\n3. 라면 – 간편하면서도 든든한 라면에 계란이나 치즈를 추가해보세요.\n4. 김밥 – 가볍게 먹기 좋고 여러 가지 재료가 들어가서 맛도 다양해요.\n5. 샌드위치 – 신선한 채소와 햄, 치즈가 들어간 샌드위치도 좋은 선택입니다.\n\n원하는 스타일이나 재료가 있으면 알려주시면 더 맞춤 추천도 가능해요!'

In [33]:
qa.memory.buffer

[HumanMessage(content='지미 카터가 누구야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='지미 카터는 민주당 출신으로 미국의 39대 대통령(1977년 ~ 1981년)을 역임한 인물입니다. 그는 1924년 10월 1일 조지아주 섬터 카운티 플레인스 마을에서 태어났습니다. 조지아 공과대학교를 졸업한 후 해군에 들어가 전함, 원자력, 잠수함의 승무원으로 일했으며, 1953년 미국 해군 대위로 예편했습니다. 이후 땅콩, 면화 등을 재배하여 많은 돈을 벌었고, "땅콩 농부"라는 별명으로 알려졌습니다.\n\n퇴임 후에는 카터 재단을 설립하여 민주주의 실현을 위한 제3세계의 선거 감시 활동, 기니 벌레에 의한 드라쿤쿠르스 질병 방재, 미국의 빈곤층 지원 활동, 사랑의 집짓기 운동, 국제 분쟁 중재 등 다양한 사회 공헌 활동을 펼쳤습니다. 또한, 그는 미국의 군사적 행동에 대해 강한 반대 입장을 보이며 평화적 해결을 주장했습니다. 대한민국과의 관계에서도 인권 문제와 주한미군 철수 문제로 한때 불편한 관계를 가졌으나, 방한을 통해 관계가 다소 회복되었습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='오늘 야식 메뉴좀 추천해줘', additional_kwargs={}, response_metadata={}),
 AIMessage(content='야식으로 먹기 좋은 메뉴를 몇 가지 추천해 드릴게요!\n\n1. **치킨**: 언제 어디서나 사랑받는 야식 메뉴입니다. 다양한 맛이 있어 선택의 폭이 넓어요.\n2. **피자**: 치즈의 풍부한 맛과 다양한 토핑이 매력적인 메뉴입니다. 여러 사람이 함께 나눠 먹기에도 좋아요.\n3. **라면**: 간편하게 끓여 먹을 수 있고, 매콤하거나 짭짤한 맛으로 밤에 먹기 좋습니다.\n4. **족발 또는 보

In [21]:
from langchain_classic.agents import Tool

tools = [
    Tool(
        name='Pinecone VectorDB',
        func=qa.run,
        description=(
            'use this tool when answering general knowledge queries to get '
            'more information about the topic'
        )
    )
]

In [22]:
from langchain_classic.agents import initialize_agent

agent = initialize_agent(
    agent='chat-conversational-react-description',
    tools=tools,
    llm=llm,
    verbose=True,
    max_iterations=3,
    early_stopping_method='generate',
    memory=conversational_memory
)

/var/folders/95/t8symb5521g2916cz4qtrsjc0000gn/T/ipykernel_18521/2766018342.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the [LangGraph documentation](https://langchain-ai.github.io/langgraph/) as well as guides for [Migrating from AgentExecutor](https://python.langchain.com/docs/how_to/migrate_agent/) and LangGraph's [Pre-built ReAct agent](https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/).
  agent = initialize_agent(


In [23]:
# 정상적인 질문해보기
agent(query)

/var/folders/95/t8symb5521g2916cz4qtrsjc0000gn/T/ipykernel_18521/547365365.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  agent(query)




> Entering new AgentExecutor chain...
```json
{
    "action": "Pinecone VectorDB",
    "action_input": "지미 카터"
}
```
Observation: 지미 카터는 미국의 39대 대통령으로, 1977년부터 1981년까지 재임했습니다. 그는 조지아주 플레인스 마을에서 태어나 조지아 공과대학교를 졸업한 후 해군에서 근무했습니다. 해군 대위로 예편한 후에는 땅콩과 면화 농사를 지으며 "땅콩 농부"라는 별명을 얻었습니다.

대통령 재임 기간 동안, 카터는 중동 평화 과정을 촉진하는 등의 외교 정책을 추진했으며, 내부적으로는 에너지 정책과 환경 보호에 중점을 두었습니다. 그러나 경제 문제와 이란 인질 위기로 인해 재선에 실패했습니다.

퇴임 후, 카터는 카터 재단을 설립하여 민주주의 실현, 질병 방재, 빈곤층 지원, 사랑의 집짓기 운동, 국제 분쟁 중재 등 다양한 사회 공헌 활동에 힘썼습니다. 그의 이러한 노력은 2002년 노벨 평화상 수상으로 이어졌습니다. 카터는 또한 미국의 군사적 행동에 대해 비판적인 입장을 견지해왔습니다.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "지미 카터는 미국의 39대 대통령으로, 1977년부터 1981년까지 재임했습니다. 그는 조지아주에서 태어나 해군에서 근무한 후 땅콩과 면화 농사를 지으며 '땅콩 농부'라는 별명을 얻었습니다. 대통령으로서 중동 평화 과정 촉진과 에너지 정책, 환경 보호에 중점을 두었으나, 경제 문제와 이란 인질 위기로 인해 재선에 실패했습니다. 퇴임 후에는 카터 재단을 설립하여 민주주의 실현, 질병 방재, 빈곤층 지원 등 다양한 사회 공헌 활동을 하며, 이러한 노력으로 2002년 노벨 평화상을 수상했습니다."
}
```

> Finished chain.


{'input': '지미 카터가 누구야?',
 'chat_history': [],
 'output': "지미 카터는 미국의 39대 대통령으로, 1977년부터 1981년까지 재임했습니다. 그는 조지아주에서 태어나 해군에서 근무한 후 땅콩과 면화 농사를 지으며 '땅콩 농부'라는 별명을 얻었습니다. 대통령으로서 중동 평화 과정 촉진과 에너지 정책, 환경 보호에 중점을 두었으나, 경제 문제와 이란 인질 위기로 인해 재선에 실패했습니다. 퇴임 후에는 카터 재단을 설립하여 민주주의 실현, 질병 방재, 빈곤층 지원 등 다양한 사회 공헌 활동을 하며, 이러한 노력으로 2002년 노벨 평화상을 수상했습니다."}

In [24]:
# 다시 쓸데없는 질문 해보기
agent("오늘 야식 메뉴 추천좀 해줘")



> Entering new AgentExecutor chain...
```json
{
    "action": "Final Answer",
    "action_input": "야식으로는 치킨, 피자, 라면, 김밥, 떡볶이와 같은 인기 메뉴가 있습니다. 기분이나 상황에 따라 가벼운 스낵에서부터 든든한 식사까지 선택할 수 있어요. 건강을 생각한다면 샐러드나 과일 같은 가벼운 메뉴도 좋습니다."
}
```

> Finished chain.


{'input': '오늘 야식 메뉴 추천좀 해줘',
 'chat_history': [HumanMessage(content='지미 카터가 누구야?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="지미 카터는 미국의 39대 대통령으로, 1977년부터 1981년까지 재임했습니다. 그는 조지아주에서 태어나 해군에서 근무한 후 땅콩과 면화 농사를 지으며 '땅콩 농부'라는 별명을 얻었습니다. 대통령으로서 중동 평화 과정 촉진과 에너지 정책, 환경 보호에 중점을 두었으나, 경제 문제와 이란 인질 위기로 인해 재선에 실패했습니다. 퇴임 후에는 카터 재단을 설립하여 민주주의 실현, 질병 방재, 빈곤층 지원 등 다양한 사회 공헌 활동을 하며, 이러한 노력으로 2002년 노벨 평화상을 수상했습니다.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'output': '야식으로는 치킨, 피자, 라면, 김밥, 떡볶이와 같은 인기 메뉴가 있습니다. 기분이나 상황에 따라 가벼운 스낵에서부터 든든한 식사까지 선택할 수 있어요. 건강을 생각한다면 샐러드나 과일 같은 가벼운 메뉴도 좋습니다.'}